In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# Problem setup: f and exact S*
# -----------------------------
def f_val(x1, x2, alpha, beta, gamma):
    return alpha*x1 + beta*x2 + gamma*x1*x2

def sobol_exact_normal(alpha, beta, gamma):
    D = alpha**2 + beta**2 + gamma**2
    return np.array([alpha**2/D, beta**2/D, gamma**2/D], dtype=float)

# -----------------------------
# Subsets (p=2): {1},{2},{1,2}
# -----------------------------
SUBSETS = [(1,), (2,), (1,2)]
Q = 3

# M^{-1} for p=2 with order [{1},{2},{1,2}]
M_inv = np.array([
    [1.0, 0.0, 0.0],   # {1}
    [0.0, 1.0, 0.0],   # {2}
    [1.0, 1.0, 1.0],   # {1,2}
], dtype=float)

# W_u = (row_u)^T row_u
W = [np.outer(M_inv[i], M_inv[i]) for i in range(Q)]

# Tangent basis for q=3: span{e1-e3, e2-e3}
B = np.array([[1.0, 0.0],
              [0.0, 1.0],
              [-1.0, -1.0]])

# -----------------------------
# Pick-Freeze sampling (N(0,1))
# -----------------------------
def sample_pf(idx, alpha, beta, gamma, rng):
    x1, x2  = rng.normal(), rng.normal()
    x1p, x2p = rng.normal(), rng.normal()

    y = f_val(x1, x2, alpha, beta, gamma)
    if idx == 0:      # {1}
        yu = f_val(x1, x2p, alpha, beta, gamma)
    elif idx == 1:    # {2}
        yu = f_val(x1p, x2, alpha, beta, gamma)
    else:             # {1,2}
        yu = y
    return y, yu

# -----------------------------
# Online mean/variance (Welford)
# -----------------------------
class OnlineMoments:
    def __init__(self):
        self.n = 0
        self.mean = 0.0
        self.M2 = 0.0
    def update(self, x):
        self.n += 1
        d = x - self.mean
        self.mean += d / self.n
        self.M2 += d * (x - self.mean)
    @property
    def var(self):
        return self.M2 / (self.n - 1) if self.n >= 2 else 0.0

# -----------------------------
# Simplex projection with lower bound
# -----------------------------
def proj_simplex(v):
    v = np.asarray(v, dtype=float)
    n = v.size
    u = np.sort(v)[::-1]
    cssv = np.cumsum(u)
    rho = np.nonzero(u - (cssv - 1) / (np.arange(n)+1) > 0)[0]
    if rho.size == 0:
        return np.ones(n)/n
    rho = rho[-1]
    theta = (cssv[rho] - 1) / (rho + 1)
    return np.maximum(v - theta, 0.0)

def proj_simplex_lower(v, a_min):
    v = np.asarray(v, dtype=float)
    q = v.size
    if q*a_min >= 1.0:
        raise ValueError("Need q*a_min < 1.")
    w = (v - a_min) / (1.0 - q*a_min)
    w = proj_simplex(w)
    return a_min + (1.0 - q*a_min)*w

# -----------------------------
# Trace criterion Tr(V^a) with tangent restriction
# -----------------------------
def Sigma_of_a(a, varY_hat):
    S = np.zeros((Q,Q), dtype=float)
    for i in range(Q):
        S += a[i] * W[i]
    return max(varY_hat, 1e-12) * S

def trace_criterion(a, varY_hat, Gamma_hat, ridge=1e-10):
    Sigma = Sigma_of_a(a, varY_hat)
    Sigma_T = B.T @ Sigma @ B + ridge*np.eye(2)
    Gamma_T = B.T @ Gamma_hat @ B
    Sinv = np.linalg.inv(Sigma_T)
    Vt = Sinv @ Gamma_T @ Sinv.T
    return float(np.trace(Vt))

# -----------------------------
# Closed-form a update from scalar Gammas (your PDF)
# a1 ∝ sqrt(G1-G12), a2 ∝ sqrt(G2-G12), a12 ∝ sqrt(G12)
# with safeguards
# -----------------------------
def a_update_closed_form(G1, G2, G12, eps=1e-12):
    w1  = np.sqrt(max(G1 - G12, eps))
    w2  = np.sqrt(max(G2 - G12, eps))
    w12 = np.sqrt(max(G12, eps))
    w = np.array([w1, w2, w12], dtype=float)
    return w / w.sum()

# -----------------------------
# One run with a policy: "unif", "cf", "num"
# -----------------------------
def run_one(
    N,
    alpha=1.0, beta=0.5, gamma=2.0,
    step_c=0.5, step_alpha=0.7,
    policy="unif",
    # common adaptation controls
    burn_in=1000,
    a_min=0.02,
    # closed-form controls
    gamma_ema_scalar=0.02,
    mix_power=0.3,    # rho_n = n^{-mix_power}
    # numeric controls
    K=500,
    eta_a=0.5,
    eps_fd=1e-4,
    gamma_ema_mat=0.01,
    seed=0
):
    rng = np.random.default_rng(seed)

    S = np.ones(Q)/Q
    S_bar = np.zeros(Q)

    a = np.ones(Q)/Q

    mom = OnlineMoments()

    # For numeric method: 
    Gamma_hat = np.zeros((Q,Q), dtype=float)

    # For closed-form method: scalar Gammas (EMA per subset)
    G = np.zeros(Q, dtype=float)  # [G1, G2, G12]

    S_star = sobol_exact_normal(alpha, beta, gamma)
    mse = np.zeros(N, dtype=float)

    for n in range(1, N+1):
        gamma_n = step_c * (n**(-step_alpha))

        # choose subset with current a
        idx = rng.choice(Q, p=a)

        # PF sample
        y, yu = sample_pf(idx, alpha, beta, gamma, rng)

        # update moments
        mom.update(y)
        m_hat = mom.mean
        varY_hat = mom.var

        yt = y - m_hat
        yut = yu - m_hat

        # compute stochastic gradient g
        row = M_inv[idx]
        MinvS_u = float(row @ S)
        scalar = yt * (yt*MinvS_u - yut)
        g = scalar * row

        # mirror update
        S *= np.exp(-gamma_n * g)
        S /= S.sum()

        # PR averaging
        S_bar += (S - S_bar) / n

        # MSE on S
        mse[n-1] = float(np.sum((S_bar - S_star)**2))

        # --------------------------------
        # Update estimators for adaptation
        # --------------------------------

        if policy == "num":
            Gamma_hat = (1.0 - gamma_ema_mat)*Gamma_hat + gamma_ema_mat*np.outer(g,g)

            if (n > burn_in) and (n % K == 0):
                def C(curr_a):
                    return trace_criterion(curr_a, varY_hat, Gamma_hat)

                grad_a = np.zeros(Q)
                for i in range(Q):
                    e = np.zeros(Q); e[i] = 1.0
                    ap = proj_simplex_lower(a + eps_fd*e, a_min)
                    am = proj_simplex_lower(a - eps_fd*e, a_min)
                    grad_a[i] = (C(ap) - C(am)) / (2.0*eps_fd)

                a = proj_simplex_lower(a - eta_a*grad_a, a_min)

        elif policy == "cf":
            # update scalar G depending on idx
            # use current PR estimate S_bar as plug-in for s
            if idx == 0:
                val = (yt**2) * (yt*S_bar[0] - yut)**2
            elif idx == 1:
                val = (yt**2) * (yt*S_bar[1] - yut)**2
            else:
                val = (yt**2) * (yt - yut)**2

            G[idx] = (1.0 - gamma_ema_scalar)*G[idx] + gamma_ema_scalar*val

            if n > burn_in:
                a_cf = a_update_closed_form(G[0], G[1], G[2])

                # exploration mixing
                rho_n = n**(-mix_power)
                a = (1.0 - rho_n)*a_cf + rho_n*(np.ones(Q)/Q)

                # enforce a_min
                a = proj_simplex_lower(a, a_min)

        else:
            # unif
            a = np.ones(Q)/Q

    return mse, S_star

# -----------------------------
# Experiment: compare 3 policies
# -----------------------------
def experiment_3methods(
    N=20000, R=30,
    alpha=1.0, beta=0.5, gamma=2.0,
    step_c=0.5, step_alpha=0.7,
    seed0=123
):
    mses_unif = []
    mses_cf = []
    mses_num = []

    for r in range(R):
        seed = seed0 + r

        mse_u, S_star = run_one(N, alpha, beta, gamma,
                                step_c, step_alpha,
                                policy="unif", seed=seed)

        mse_cf, _ = run_one(N, alpha, beta, gamma,
                            step_c, step_alpha,
                            policy="cf", seed=seed)

        mse_num, _ = run_one(N, alpha, beta, gamma,
                             step_c, step_alpha,
    mses_unif = np.array(mses_unif)
    mses_cf = np.array(mses_cf)
    mses_num = np.array(mses_num)

    mean_u = mses_unif.mean(axis=0)
    mean_cf = mses_cf.mean(axis=0)
    mean_num = mses_num.mean(axis=0)

                             policy="num", seed=seed)

        mses_unif.append(mse_u)
        mses_cf.append(mse_cf)
        mses_num.append(mse_num)

    se_u = mses_unif.std(axis=0, ddof=1)/np.sqrt(R)
    se_cf = mses_cf.std(axis=0, ddof=1)/np.sqrt(R)
    se_num = mses_num.std(axis=0, ddof=1)/np.sqrt(R)

    x = np.arange(1, N+1)

    plt.figure(figsize=(7,4))
    plt.loglog(x, mean_u, label="Mirror-Unif")
    plt.loglog(x, mean_cf, label="Mirror-Adapt (formule fermée)")
    plt.loglog(x, mean_num, label="Mirror-Adapt (min Tr(V^a))")
    plt.xlim(left=500)

    plt.fill_between(x, np.maximum(mean_u - 1.96*se_u, 1e-18), mean_u + 1.96*se_u, alpha=0.15)
    plt.fill_between(x, np.maximum(mean_cf - 1.96*se_cf, 1e-18), mean_cf + 1.96*se_cf, alpha=0.15)
    plt.fill_between(x, np.maximum(mean_num - 1.96*se_num, 1e-18), mean_num + 1.96*se_num, alpha=0.15)

    plt.loglog(x, 1/x, "--", label="1/n")
    plt.xlabel("n")
    plt.ylabel(r"MSE $\|\bar S_n - S^*\|^2$")
    plt.title(f"Comparaison 3 méthodes (N(0,1)), R={R}\n"
              f"alpha={alpha}, beta={beta}, gamma={gamma}, step={step_c} n^-{step_alpha}")
    plt.grid(True, which="both", ls="--", alpha=0.3)
    plt.legend()
    plt.show()

    print("S* exact =", S_star)
    print("Final mean MSE Unif =", mean_u[-1])
    print("Final mean MSE CF   =", mean_cf[-1])
    print("Final mean MSE NUM  =", mean_num[-1])

if __name__ == "__main__":
    experiment_3methods(N=500000, R=30)